# **Import Libraries**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, EarlyStoppingCallback, Trainer
import evaluate
from sklearn.metrics import confusion_matrix, classification_report

# Summarization
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from transformers import pipeline     # for BART model
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func

# Aspect Extraction
import spacy
import torch

nltk.download("punkt")
nltk.download("stopwords")
nltk.download('punkt_tab')

# **Loading Data**

In [ ]:
df = pd.read_csv('combine (1).csv', sep=',')

# **EDA (Exploratory Data Analysis)**

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# RENAME COLUMN
df = df.rename(columns={'placeInfo/name':'Hotel'})

In [ ]:
# DELETE UNNECESSARY COLUMN
keep_cols = ['Hotel', 'rating', 'text', 'lang']
df = df[keep_cols]

In [ ]:
df.info()

In [ ]:
#CHECK NULL/MISSING VALUES
df.isnull().sum()

In [ ]:
# CHECK DUPLICATED DATA
df.duplicated().sum()

In [ ]:
# DISPLAY THE DUPLICATED ROWS
df[df.duplicated(keep=False)]

In [ ]:
# DROP DUPLICATED DATA & RESET THE INDEX
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# SELECT ONLY ENGLISH LANGUAGE REVIEW
df = df[df['lang'] == 'en'].reset_index(drop=True)

In [ ]:
# CHECK HOTEL DISTRIBUTION
df['Hotel'].value_counts()

In [ ]:
# HOTEL DISTRIBUTION VISUALIZATION
plt.figure()
df['Hotel'].value_counts().plot(kind='bar')
plt.title("Hotel Distributions")
plt.show()

In [ ]:
# CHECK RATING DISTRIBUTION
df['rating'].value_counts()

In [ ]:
# RATING DISTRIBUTION VISUALIZATION
plt.figure()
df['rating'].value_counts().plot(kind='bar')
plt.title("Rating Distribution")
plt.show()

In [ ]:
# CREATE SENTIMENT LABELS
df['Label'] = np.where(df['rating'] > 3, 'POSITIVE', 'NEGATIVE')

In [ ]:
df.head(10)

In [ ]:
# CHECK LABEL DISTRIBUTION
df['Label'].value_counts()

# **Text Cleaning**

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z0-9,.!? ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# ENCODE LABELS TO NUMERIC
df['label_encoded'] = df['Label'].map({"NEGATIVE" : 0, "POSITIVE" : 1})
df['label_encoded'].value_counts()

# **Dataset Split**

In [ ]:
# 80% Train, 20% Temp (Validation + Test)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label_encoded'], random_state=42)

# Split temp into validation and test (50% of temp each -> 10% of original each)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_encoded'], random_state=42)

# Print shapes
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# **Tokenization & Dataset Preparation**

In [ ]:
# Convert pandas dataframe into Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train" : train_dataset,
    "validation" : val_dataset,
    "test" : test_dataset
})

dataset

In [ ]:
# Load the tokenizer
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Define the tokenization function
def tokenize_function(batch):
  return tokenizer(
      batch['clean_text'],
      padding="max_length",   # pad all to 512 tokens
      truncation=True,        # truncate long reviews
      max_length=256          # 256 is enough for hotel reviews (faster)
  )

In [ ]:
# Apply tokenization to the Dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
tokenized_dataset

In [ ]:
# Remove columns that the model doesn't need
# Transformers only need:
# - input_ids
# - attention_mask
# - label
tokenized_dataset = tokenized_dataset.remove_columns([
    'Hotel',
    'rating',
    'text',
    'lang',
    'Label',
    'clean_text',
    '__index_level_0__',
    'token_type_ids'
])

In [ ]:
# Rename label_encoded -> labels
# HuggingFace Trainer requires the target column to be named labels.
tokenized_dataset = tokenized_dataset.rename_column('label_encoded', 'labels')

In [ ]:
tokenized_dataset

In [ ]:
# Set the Dataset format to PyTorch tensors
tokenized_dataset.set_format('torch')

# **Model Training**

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device: ", DEVICE)

In [ ]:
# Load the model
num_labels = 2    # POSITIVE & NEGATIVE

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels = num_labels
).to(DEVICE)

In [ ]:
# Define Training Arguments
training_args = TrainingArguments(
    output_dir="deberta_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,    # can be high because early stopping will automatically stop it.
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    seed=42,
    report_to = "none",
    
    # To make training faster & lower GPU memory usage
    fp16=True,              # <-- enable mixed precision training
    fp16_full_eval=True     # <-- use FP16 during evaluation too

)

In [ ]:
# Define Evaluation Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)

    return {
        # evaluate.compute() returns a dictionary (e.g., {"accuracy": 0.89}). Thus, we extract it using the dictionary key.
        "accuracy": accuracy.compute(predictions=preds, references=labels)['accuracy'],
        "precision": precision.compute(predictions=preds, references=labels)['precision'],
        "recall": recall.compute(predictions=preds, references=labels)['recall'],
        "f1": f1.compute(predictions=preds, references=labels)['f1']
    }

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# Train the model
trainer.train()

In [ ]:
# Check the model's best checkpoint
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

# **Model Evaluation**

In [ ]:
# Predict the test set

# Get raw predictions
predictions = trainer.predict(tokenized_dataset["test"])
print(predictions)

# Convert logits to class labels (0 or 1)
y_pred = np.argmax(predictions.predictions, axis=1)

# True labels
y_true = predictions.label_ids


In [ ]:
# Evaluate on the test set
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NEG','POS'], yticklabels=['NEG','POS'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print(classification_report(y_true, y_pred, target_names=['NEGATIVE','POSITIVE']))

# **Save Model**

In [ ]:
# Save Model & Tokenizer
trainer.save_model("best_sentiment_model")
tokenizer.save_pretrained("best_sentiment_model")

In [ ]:
# Load the saved model
best_model = AutoModelForSequenceClassification.from_pretrained("best_sentiment_model")
tokenizer = AutoTokenizer.from_pretrained("best_sentiment_model")

# **Summarization**

In [ ]:
# Group reviews by hotel
grouped_reviews = df.groupby('Hotel')['clean_text'].apply(list).reset_index()
grouped_reviews.head()

In [ ]:
# Combine reviews into one long text
grouped_reviews['combined_reviews'] = grouped_reviews['clean_text'].apply(lambda x: " ".join(x))
grouped_reviews.head()

In [ ]:
stop_words = stopwords.words("english")

In [ ]:
def simple_extractive_summary(text, num_sentences=2):
  sentences = sent_tokenize(text)

  # Create word frequency
  words = word_tokenize(text.lower())
  words = [word for word in words if word not in stop_words and word.isalnum()]
  freq = Counter(words)

  # Score each sentence
  sentence_scores = {}
  for sent in sentences:
    for word in word_tokenize(sent.lower()):
      if word in freq:
        sentence_scores[sent] = sentence_scores.get(sent, 0) + freq[word]

  # Pick top N sentences
  top_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)[:num_sentences]

  return " ".join(top_sentences)

In [ ]:
grouped_reviews["summary"] = grouped_reviews["combined_reviews"].apply(
    lambda x: simple_extractive_summary(x, num_sentences=3)
)

grouped_reviews.head()

In [ ]:
# Load summarization model
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    tokenizer="facebook/bart-large-cnn",
    device = 0 if torch.cuda.is_available() else -1
)

In [ ]:
def safe_bart_summarize(text):
    try:
        # Hard limit input size
        text = text[:1000]

        summary = summarizer(
            text,
            max_length=60,
            min_length=20,
            do_sample=False
        )
        return summary[0]["summary_text"]

    except Exception as e:
        return "Summary failed"

In [ ]:
grouped_reviews["bart_summary"] = grouped_reviews["combined_reviews"].apply(safe_bart_summarize)
grouped_reviews[["Hotel", "bart_summary"]].head()

# **Summarization Evaluation**

In [ ]:
# ROUGE score
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for simple_extractive_sum, bart_sum in zip(grouped_reviews['summary'], grouped_reviews['bart_summary']):
  score = scorer.score(simple_extractive_sum, bart_sum)

  rouge1_scores.append(score['rouge1'].fmeasure)
  rouge2_scores.append(score['rouge2'].fmeasure)
  rougeL_scores.append(score['rougeL'].fmeasure)

print("ROUGE-1:", np.mean(rouge1_scores))
print("ROUGE-2:", np.mean(rouge2_scores))
print("ROUGE-L:", np.mean(rougeL_scores))

The results indicate that the semantic overlap between the abstractive summaries (BART) and the extractive summaries is relatively low.

This is expected because:
- The BART model tends to generate more **paraphrased** and **abstract** summaries, rather than copying exact phrases from the source.

- Extractive summaries **copying exact phrases** from the source.

- ROUGE evaluates the **similarity** of the summarized sentences, **not the quality**.

Despite the low ROUGE values, manual inspection shows that the generated summaries still retain the **main ideas** and **sentiments** of the hotel reviews, indicating that the system is capable of producing **meaningful abstractive summaries**.

In [ ]:
# BERTScore (Semantic Similarity)
P, R, F1 = bert_score_func(
    grouped_reviews["bart_summary"].tolist(),
    grouped_reviews["summary"].tolist(),
    lang="en",
    device = "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
print("BERTScore F1:", F1.mean().item())

To address **ROUGE score limitation**, **BERTScore** was used to **measure semantic similarity** between the summaries. Unlike ROUGE, which relies on surface-level word overlap (similarities), **BERTScore computes contextual similarity using BERT embeddings**, making it more suitable for evaluating paraphrased summaries. The summarization model achieved a high BERTScore **F1 score** of **0.82**, indicating that the generated summaries **preserved the core meaning of the original reviews despite using different wording**. This demonstrates that the abstractive summarization model **successfully** captured the main information from user reviews.

In [ ]:
summarizer.model.save_pretrained("bart_model")
summarizer.tokenizer.save_pretrained("bart_model")

# **Aspect Extraction**

In [ ]:
spacy.cli.download('en_core_web_sm')

In [ ]:
# Load spaCy model
nlp = spacy.load('en_core_web_sm')

In [ ]:
HOTEL_ASPECTS = {
    "room", "rooms", "bed", "bathroom", "toilet", "shower", "staff", "service", "location",
    "pool", "pools", "wifi", "internet", "cleanliness", "food", "breakfast", "dinner",
    "price", "view", "beach", "air conditioning", "ac", "facility", "facilities", "restaurant",
    "budget", "night", "stay", "experience", "floor", "size", "space", "comfort",
    "amenity", "beverage", "drink", "quality", "access", "gym", "spa", "park", "carpark",
    "reception", "management", "value", "cost", "balcony", "hotel", "city", "towels", "towel",
    "atmosphere"
}

In [ ]:
# Extract Aspects (Nouns) From Summary
# Nouns (NOUN)
# Proper nouns (PROPN)

def extract_aspects(text):
  doc = nlp(text)

  aspects = []
  for token in doc:
    if token.pos_ in ['NOUN', 'PROPN'] and token.text in HOTEL_ASPECTS:
      aspects.append(token.lemma_.lower())  # use lemma to group "rooms" -> "room"

  return list(set(aspects))   # remove duplicates

In [ ]:
# Extract Aspects Sentences
def get_aspect_sentences(text, aspects):
  sentences = sent_tokenize(text)

  aspect_map = {}
  for aspect in aspects:
    for sentence in sentences:
      if aspect in sentence.lower():
        aspect_map.setdefault(aspect, []).append(sentence)

  return aspect_map

# **Sentiment Classification**

In [ ]:
def predict_sentiment(sentence):
  # Tokenize the sentence → convert to input_ids + attention_mask
  inputs = tokenizer(sentence, return_tensors='pt', truncation=True)
  # Feed into model → get raw logits
  outputs = best_model(**inputs)
  # Convert logits into probabilities
  probs = torch.softmax(outputs.logits, dim=1)
  # Pick highest probability class
  pred = torch.argmax(probs, dim=1).item()
  # Convert class ID to label name
  return "positive" if pred == 1 else "negative"

In [ ]:
def aspect_sentiment_pipeline(summary_text):
    aspects = extract_aspects(summary_text)
    aspect_sentences = get_aspect_sentences(summary_text, aspects)

    results = {}

    for asp, sentences in aspect_sentences.items():
        # classify each sentence about this aspect
        sentiments = [predict_sentiment(s) for s in sentences]

        # majority vote (if multiple sentences mention the same aspect)
        final_sentiment = max(set(sentiments), key=sentiments.count)

        results[asp] = final_sentiment

    return results

In [ ]:
print(aspect_sentiment_pipeline("The room is good. The food is bad. The room is small. The room is clean. The pool is big."))

In [ ]:
grouped_reviews['aspect_sentiment'] = grouped_reviews['bart_summary'].apply(aspect_sentiment_pipeline)

In [ ]:
grouped_reviews[['Hotel', 'aspect_sentiment']].head()

In [ ]:
grouped_reviews.to_csv("grouped_reviews.csv", index=False)